In [1]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything

# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model - This requries internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


In [5]:
from pathlib import Path
from PIL import Image
import numpy as np
import torch

from mapanything.models import MapAnything
from mapanything.utils.image import preprocess_inputs

BASE = Path("/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3")
ALL_RGB = sorted((BASE / "linearPNG").glob("*.png"))

# Pick frames 0, 120, 240 (0-based indexing)
frame_idxs = [0, 120, 240]
selected_rgb = [ALL_RGB[i] for i in frame_idxs if i < len(ALL_RGB)]

K = np.array([[987.46, 0.0, 830.36],
              [0.0, 987.46, 644.75],
              [0.0, 0.0, 1.0]], dtype=np.float32)

views = []
for rgb_path in selected_rgb:
    depth_path = BASE / "decoded_npy" / f"{rgb_path.stem}.npy"
    if not depth_path.exists():
        continue

    rgb = np.array(Image.open(rgb_path).convert("RGB"))
    depth = np.load(depth_path).astype(np.float32)

    views.append({
        "img": rgb,
        "intrinsics": K,
        "depth_z": depth,
        "is_metric_scale": torch.tensor([True]),
    })

processed_views = preprocess_inputs(views)


In [6]:
# Run inference with any combination of inputs
predictions = model.infer(
    processed_views,                  # Any combination of input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    # Control which inputs to use/ignore
    # By default, all inputs are used when provided
    # If is_metric_scale flag is not provided, all inputs are assumed to be in metric scale
    ignore_calibration_inputs=False,
    ignore_depth_inputs=False,
    ignore_pose_inputs=False,
    ignore_depth_scale_inputs=False,
    ignore_pose_scale_inputs=False,
)

In [7]:
import numpy as np
import open3d as o3d

pred = predictions[0]  # first view

pts3d = pred["pts3d"].squeeze().cpu().numpy()            # (H, W, 3)
colors = pred["img_no_norm"].squeeze().cpu().numpy()     # (H, W, 3) in [0, 1]
mask = pred["mask"].squeeze().cpu().numpy().astype(bool) # validity mask

xyz = pts3d[mask]
rgb = colors[mask]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)
pcd.colors = o3d.utility.Vector3dVector(rgb)

o3d.visualization.draw_geometries([pcd], window_name="MapAnything Reconstruction")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
